In [0]:
df = spark.read.table("teams.sensorx.df_sx_800")

#display(df)


do horizon for the failiures - create a new column with failiure horizon based on some horizon H


convert timestamp to datetime in pandas
delta  - add as a feature


Altering data for models:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.sql.functions import col

# --- 1) Lag features ---
lags = [1, 2, 3]
w = Window.orderBy("timeStamp")

# Filter for specific deviceId
df2 = df.filter(F.col("properties_deviceId") == "3a687fe1-e2bc-4bbf-f64c-08dad2cde7be")

for L in lags:
    df2 = df2.withColumn(f"payload_xrayController_filamentCurrent{L}", F.lag("payload_xrayController_filamentCurrent", L).over(w))
    df2 = df2.withColumn(f"payload_xrayController_temperature{L}", F.lag("payload_xrayController_temperature", L).over(w))
    df2 = df2.withColumn(f"payload_xrayController_tubeCurrent{L}", F.lag("payload_xrayController_tubeCurrent", L).over(w))
    df2 = df2.withColumn(f"payload_xrayController_tubeVoltage{L}", F.lag("payload_xrayController_tubeVoltage", L).over(w))

df2 = df2.drop(
    "serialNumber",
    "payload_fault_faultName",
    "properties_deviceId",
    "GeneratorType"
)

feature_cols = [c for c in df2.columns if c not in ("payload_fault_state", "timeStamp", "rn")]

df2 = df2.na.drop(subset=feature_cols)

# --- 2) Chronological split ---
df2 = df2.withColumn("rn", F.row_number().over(w))
total = df2.count()
cut = int(total * 0.8)

train = df2.filter(F.col("rn") <= cut)
test  = df2.filter(F.col("rn") >  cut)

# --- 3) Assemble features ---
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

train = assembler.transform(train).select("features", F.col("payload_fault_state").cast('integer').alias("label"))
test  = assembler.transform(test).select("features", F.col("payload_fault_state").cast('integer').alias("label"))




Logistic Regression with lag:

In [0]:
# Logistic regression:
lr = LogisticRegression(maxIter=100, regParam=0.0, elasticNetParam=0.0)

# Fit the model on the 80% training data
lrModel = lr.fit(train)

# Print the coefficients and intercept for logistic regression
print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))

LRpredictions = lrModel.transform(test)
#display(predictions.select("prediction", "label", "features"))

from pyspark.ml.classification import LogisticRegression

# Extract the summary from the returned LogisticRegressionModel instance trained
# in the earlier example
LRtrainingSummary = lrModel.summary

# Obtain the receiver-operating characteristic as a dataframe and areaUnderROC.
LRtrainingSummary.roc.show()
print("areaUnderROC: " + str(LRtrainingSummary.areaUnderROC))

print("precisionByLabel: " + str(LRtrainingSummary.precisionByLabel))

# Set the model threshold to maximize F-Measure
fMeasure = LRtrainingSummary.fMeasureByThreshold
maxFMeasure = fMeasure.groupBy().max('F-Measure').select('max(F-Measure)').head()
bestThreshold = fMeasure.where(fMeasure['F-Measure'] == maxFMeasure['max(F-Measure)']) \
    .select('threshold').head()['threshold']
lr.setThreshold(bestThreshold)

Random forest with lag:

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


# --- 4) RandomForest (Spark ML) ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

RFmodel = rf.fit(train)
RFpred = RFmodel.transform(test)

# Extract the summary from the returned RandomForestClassifier instance trained
# in the earlier example
RFtrainingSummary = RFmodel.summary

# Obtain the receiver-operating characteristic as a dataframe and areaUnderROC.
RFtrainingSummary.roc.show()
print("recallByLabel: " + str(RFtrainingSummary.recallByLabel))

print("precisionByLabel: " + str(RFtrainingSummary.precisionByLabel))

RFpred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# --- 5) Simple evaluation (F1) ---
eval_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)



Gradient Boosted trees:


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Index labels, adding metadata to the label column.
labelIndexer = StringIndexer(inputCol="label", outputCol="indexedLabel").fit(train)

# Automatically identify categorical features, and index them.
featureIndexer = VectorIndexer(inputCol="features", outputCol="indexedFeatures", maxCategories=4).fit(train)

# Train a GBT model.
gbt = GBTClassifier(labelCol="indexedLabel", featuresCol="indexedFeatures", maxIter=10)

# Chain indexers and GBT in a Pipeline
pipeline = Pipeline(stages=[labelIndexer, featureIndexer, gbt])

# Train model.  This also runs the indexers.
model = pipeline.fit(train)

# Make predictions.
gbtpredictions = model.transform(test)

# Select example rows to display.
display(gbtpredictions.select("prediction", "indexedLabel", "features").limit(5))

# Select (prediction, true label) and compute test error
evaluator = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(gbtpredictions)
print("Test Error = %g" % (1.0 - accuracy))

gbtModel = model.stages[2]
print(gbtModel)  # summary only

In [0]:
# Logistic Regression:
LRmatrix = (
    LRpredictions.groupBy("label")
        .pivot("prediction")
        .count()
        .orderBy("label")
)
matrix.show()
print("areaunderROC: " + str(LRtrainingSummary.areaUnderROC))

# Random Forest Classifier:
RFmatrix = (
    RFpred.groupBy("label")
        .pivot("prediction")
        .count()
        .orderBy("label")
)
RFmatrix.show()

print("areaunderROC: " + str(RFtrainingSummary.areaUnderROC))

# Gradient Boosted Trees Classifier:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

GBTmatrix = (
    gbtpredictions.groupBy("label")
        .pivot("prediction")
        .count()
        .orderBy("label")
)
GBTmatrix.show()

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",  # or "probability"
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(gbtpredictions)
print("AUC =", auc)

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# =====================================================
# 1. Create ONE evaluator that works for ALL models
# =====================================================
auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# =====================================================
# 2. Confusion matrix function (clean & consistent)
# =====================================================
def confusion(pred_df):
    return (
        pred_df.groupBy("label")
            .pivot("prediction", [0.0, 1.0])   # force same columns
            .count()
            .fillna(0)                         # replace nulls with 0
            .orderBy("label")
    )

# =====================================================
# 3. Evaluate Logistic Regression
# =====================================================
print("\nLogistic Regression")
LRmatrix = confusion(LRpredictions)
LRmatrix.show()
LR_auc = auc_eval.evaluate(LRpredictions)
print("AUC =", LR_auc)

# =====================================================
# 4. Evaluate Random Forest
# =====================================================
print("\nRandom Forest")
RFmatrix = confusion(RFpred)
RFmatrix.show()
RF_auc = auc_eval.evaluate(RFpred)
print("AUC =", RF_auc)

# =====================================================
# 5. Evaluate Gradient Boosted Trees
# =====================================================
print("\nGradient Boosted Trees")
GBTmatrix = confusion(gbtpredictions)
GBTmatrix.show()
GBT_auc = auc_eval.evaluate(gbtpredictions)
print("AUC =", GBT_auc)